# H07 — 让电脑找到这台工具

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H07 — 让电脑找到这台工具。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H07-wifi.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

LED 工作了，但电脑还不知道设备在哪。连接 AP、拿到 IP、找到服务是三件事；断网以后不能只靠重启。

前置：H06；可控制的 2.4 GHz AP。

## 本次够用的知识

Wi-Fi 关联与获得 IP 不等于应用可达。DHCP 分配地址，DNS 把名字变为地址；mDNS 在局域网发现名称/服务。事件回调要短，重连用延迟退避；阻塞回调会拖累整个系统。经典 ESP32 仅支持 2.4 GHz Wi-Fi。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 先排除网络拓扑 · Guided

确认 2.4 GHz AP、电脑和 ESP32 位于可互通 LAN；访客隔离、校园网认证与企业认证不是这个 WPA2-PSK 最小示例的范围。
给网络模块定义 disconnected/connecting/has_ip 状态；设备密码只在本地 menuconfig，sdkconfig 被忽略，不提交，不在日志输出。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 事件驱动连接 · Guided

将 wifi_link.hpp/.cpp 放 main（H06 后可移 network 组件），main/Kconfig.projbuild 加入本课菜单。SRCS 登记 wifi_link.cpp，REQUIRES 增加 esp_wifi esp_event esp_netif esp_timer nvs_flash。
menuconfig 的 Academy local network 填 SSID/密码。本示例初始值空，故意不能自动连接陌生网络。读取 CONNECT→IP 日志，再把 link 服务增量接进持续工程，初始化只能执行一次。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H07/examples/wifi_link.hpp`。目标：`main/wifi_link.hpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。

In [ ]:
%%esp32_example --target main/wifi_link.hpp
#pragma once
#include "esp_err.h"
#include "freertos/FreeRTOS.h"
esp_err_t wifi_link_start();
bool wifi_link_wait(TickType_t timeout);


示例来源：`step2/lessons/H07/examples/wifi_link.cpp`。目标：`main/wifi_link.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/wifi_link.cpp
#include "wifi_link.hpp"
#include <cstring>
#include "esp_event.h"
#include "esp_log.h"
#include "esp_netif.h"
#include "esp_timer.h"
#include "esp_wifi.h"
#include "freertos/event_groups.h"
#include "nvs_flash.h"
#include "sdkconfig.h"

static EventGroupHandle_t state;
static esp_timer_handle_t retry_timer;
static unsigned retries = 0;
static constexpr EventBits_t kHasIp = BIT0;
static void retry(void*) {
    const esp_err_t err = esp_wifi_connect();
    if (err != ESP_OK) {
        ESP_LOGW("wifi", "connect rejected: %s", esp_err_to_name(err));
    }
}
static void on_event(void*, esp_event_base_t base, int32_t id, void* data) {
    if (base == WIFI_EVENT && id == WIFI_EVENT_STA_START) {
        retry(nullptr);
    } else if (base == WIFI_EVENT && id == WIFI_EVENT_STA_DISCONNECTED) {
        xEventGroupClearBits(state, kHasIp);
        const auto* event = static_cast<wifi_event_sta_disconnected_t*>(data);
        const unsigned seconds = 1U << (retries < 5 ? retries++ : 5);
        ESP_LOGW("wifi", "disconnected reason=%u; retry in %us", event->reason,
                 seconds);
        esp_timer_stop(retry_timer); // Already stopped is harmless here.
        ESP_ERROR_CHECK(esp_timer_start_once(retry_timer, uint64_t(seconds) * 1000000));
    } else if (base == IP_EVENT && id == IP_EVENT_STA_GOT_IP) {
        retries = 0;
        esp_timer_stop(retry_timer);
        auto* event = static_cast<ip_event_got_ip_t*>(data);
        ESP_LOGI("wifi", "IP=" IPSTR, IP2STR(&event->ip_info.ip));
        xEventGroupSetBits(state, kHasIp);
    }
}
esp_err_t wifi_link_start() {
    if (std::strlen(CONFIG_ACADEMY_WIFI_SSID) == 0) {
        ESP_LOGE("wifi",
                 "Set local Wi-Fi credentials in menuconfig; never commit them.");
        return ESP_ERR_INVALID_ARG;
    }
    // Do not automatically erase a user's NVS on initialization failure.
    ESP_ERROR_CHECK(nvs_flash_init());
    ESP_ERROR_CHECK(esp_netif_init());
    ESP_ERROR_CHECK(esp_event_loop_create_default());
    if (!esp_netif_create_default_wifi_sta()) {
        return ESP_ERR_NO_MEM;
    }
    state = xEventGroupCreate();
    if (!state) {
        return ESP_ERR_NO_MEM;
    }
    esp_timer_create_args_t timer{};
    timer.callback = retry;
    timer.name = "wifi_retry";
    ESP_ERROR_CHECK(esp_timer_create(&timer, &retry_timer));
    wifi_init_config_t init = WIFI_INIT_CONFIG_DEFAULT();
    ESP_ERROR_CHECK(esp_wifi_init(&init));
    ESP_ERROR_CHECK(
        esp_event_handler_register(WIFI_EVENT, ESP_EVENT_ANY_ID, on_event, nullptr));
    ESP_ERROR_CHECK(
        esp_event_handler_register(IP_EVENT, IP_EVENT_STA_GOT_IP, on_event, nullptr));
    wifi_config_t cfg{};
    static_assert(sizeof(CONFIG_ACADEMY_WIFI_SSID) - 1 <= sizeof(cfg.sta.ssid));
    static_assert(sizeof(CONFIG_ACADEMY_WIFI_PASSWORD) - 1 <= sizeof(cfg.sta.password));
    std::memcpy(cfg.sta.ssid, CONFIG_ACADEMY_WIFI_SSID,
                sizeof(CONFIG_ACADEMY_WIFI_SSID) - 1);
    std::memcpy(cfg.sta.password, CONFIG_ACADEMY_WIFI_PASSWORD,
                sizeof(CONFIG_ACADEMY_WIFI_PASSWORD) - 1);
    cfg.sta.threshold.authmode = WIFI_AUTH_WPA2_PSK;
    ESP_ERROR_CHECK(esp_wifi_set_storage(WIFI_STORAGE_RAM));
    ESP_ERROR_CHECK(esp_wifi_set_mode(WIFI_MODE_STA));
    ESP_ERROR_CHECK(esp_wifi_set_config(WIFI_IF_STA, &cfg));
    return esp_wifi_start();
}
bool wifi_link_wait(TickType_t timeout) {
    return state &&
           (xEventGroupWaitBits(state, kHasIp, pdFALSE, pdTRUE, timeout) & kHasIp);
}


示例来源：`step2/lessons/H07/examples/main.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "wifi_link.hpp"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    ESP_ERROR_CHECK(wifi_link_start());
    while (true) {
        ESP_LOGI("status", "has_ip=%d", wifi_link_wait(pdMS_TO_TICKS(1000)));
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


### 本课配置文件

来源：`step2/lessons/H07/examples/Kconfig.projbuild` → `main/Kconfig.projbuild`。
已有文件时合并 menu，不覆盖自己的 GPIO/采样配置。这里只填写 Wi-Fi；MQTT/OTA 字段留到相应课程。
下列是配置文本，放到工程文件，不当作 C++/Python 运行。默认空凭据会明确失败，不会假装连上。

```kconfig
menu "Academy local network"
    config ACADEMY_WIFI_SSID
        string "Wi-Fi SSID (local secret)"
        default ""
    config ACADEMY_WIFI_PASSWORD
        string "Wi-Fi password (local secret)"
        default ""
    config ACADEMY_MQTT_URI
        string "MQTT broker URI (private LAN only for mqtt://)"
        default "mqtt://192.168.1.2:1883"
    config ACADEMY_OTA_URL
        string "HTTPS firmware URL"
        default "https://example.invalid/firmware.bin"
endmenu
```


## 切断 AP 再恢复 · Modify

在你可控制的 AP 上断开/恢复网络，记录 disconnect reason、退避间隔和新 IP。观察计时与按钮是否仍在本地工作。
SSID 或密码错误不会靠无限快速重连解决；分别测试错误密码、AP 不在和重新可用，用日志做判断。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 名字与地址 · Build

在获得 IP 后用 lwIP getaddrinfo 查询一个明确可访问的名称，记录错误码、地址族和释放结果；DNS 失败不能当作 Wi-Fi 失败。
为局域网名称使用受维护的 espressif/mdns 组件，先查与 v5.5.1 兼容的版本、添加依赖并提交 lock，再调用 mdns_init、mdns_hostname_set、mdns_service_add。这是构建练习，不把未锁版本的 API 当已验证；IP 访问是必备恢复路径。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 不要在事件回调睡觉 · Debug

暂时在 WIFI_EVENT_STA_DISCONNECTED 分支加入 3 s 的 vTaskDelay，观察事件处理受阻，再恢复 Timer 延迟重连。
若电脑访问失败，按 AP 关联 → IP/网段 → DNS/mDNS → port → HTTP 顺序排查；ping 不响应不能唯一证明设备离线。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 网络状态接入业务 · Build

需求：断网本地按钮/计时继续工作；重连有最大间隔；连接状态可观测；恢复后协议重新就绪；DNS 超时有上界。
网络回调只送事件，Controller 统一决定状态灯。测试 repeated disconnect、IP 改变和启动无 AP；不以 log=connected 验收端到端访问。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 网络诊断工具雏形 · Open

设计显示“无线未连接/尚无 IP/服务故障”的三种反馈，解释不使用一种红灯概括所有问题的原因。
H08 将用电脑请求真实 API；把 IP、服务端口与当前固件版本写入记录。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 实际关联、IP 与应用可达分别记录
- [ ] 断 AP 后本地功能仍在，恢复有日志
- [ ] 凭据未进入 Git/日志，DNS 错误单独报告

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H07',
    [
        '实际关联、IP 与应用可达分别记录',
        '断 AP 后本地功能仍在，恢复有日志',
        '凭据未进入 Git/日志，DNS 错误单独报告',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H07/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/network/esp_wifi.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/network/esp_wifi.html)